# Synthetic Data Generator Pipeline
### Step 6 : Producer Queue Manager


In [1]:
import os

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
)

from utils.synthetic.pipeline.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    get_send_queue_status_counts,
)

In [2]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")

SIMULATION_TABLE_NAME = "simulation_state_control"
SEND_QUEUE_TABLE_NAME = "synthetic_sensor_messages_send_queue"

PRODUCER_TOPIC = os.getenv(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
)

PRODUCER_WORKER_ID = os.getenv(
    "SYNTHETIC_PRODUCER_WORKER_ID",
    "producer_worker_test_001",
)

OBSERVATION_BATCH_SIZE = int(os.getenv("SYNTHETIC_OBSERVATION_BATCH_SIZE", "500"))
PRODUCER_POLL_SECONDS = float(os.getenv("SYNTHETIC_PRODUCER_POLL_SECONDS", "0.0"))
PRODUCER_MAX_SEND_ATTEMPTS = int(os.getenv("SYNTHETIC_PRODUCER_MAX_SEND_ATTEMPTS", "3"))

CONTROL_OWNER_ROLE = "kafka_producer"
APPLY_OWNER_AND_GRANTS_FLAG = False

print("Producer queue manager config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("send queue table:", SEND_QUEUE_TABLE_NAME)
print("producer topic:", PRODUCER_TOPIC)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("apply owner/grants:", APPLY_OWNER_AND_GRANTS_FLAG)

Producer queue manager config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
send queue table: synthetic_sensor_messages_send_queue
producer topic: pump.telemetry.synthetic
observation batch size: 500
apply owner/grants: False


---

In [13]:

engine = get_engine_from_env()


---

In [4]:
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    ).loc[0, "sensor_count"]
)

PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Derived producer batch sizing")
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("producer batch size:", PRODUCER_BATCH_SIZE)

Derived producer batch sizing
number of sensors: 52
observation batch size: 500
producer batch size: 26000


---

In [5]:
control_table_name = ensure_simulation_state_control_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
    owner_role=CONTROL_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS_FLAG,
)

print("Ensured control table:", control_table_name)

Ensured control table: simulation_state_control


---

In [6]:
upsert_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    is_enabled=True,
    producer_topic=PRODUCER_TOPIC,
    producer_batch_size=PRODUCER_BATCH_SIZE,
    producer_poll_seconds=PRODUCER_POLL_SECONDS,
    max_send_attempts=PRODUCER_MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)

print("Upserted control row.")

Upserted control row.


---

In [7]:
control_row = read_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)

display(control_row)

{'dataset_id': 'pump_synthetic_v1',
 'run_id': 'synthetic_run_001',
 'is_enabled': True,
 'producer_topic': 'pump.telemetry.synthetic',
 'producer_batch_size': 26000,
 'producer_poll_seconds': 0.0,
 'max_send_attempts': 3,
 'updated_at': Timestamp('2026-05-26 17:46:22.693908+0000', tz='UTC'),
 'created_at': Timestamp('2026-05-24 01:08:02.623078+0000', tz='UTC')}

---

In [8]:
ensure_send_queue_runtime_columns(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Ensured send queue runtime columns and indexes.")

Ensured send queue runtime columns and indexes.


---

In [9]:
queue_health_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS unclaimed_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS claimed_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_timestamp_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(queue_health_dataframe)

,queue_status,row_count,unclaimed_count,claimed_count,unsent_count,sent_timestamp_count
0,pending,11700000,11700000,0,11700000,0


---

In [10]:
producer_pickup_explain_dataframe = read_sql_dataframe(
    engine,
    f"""
    EXPLAIN ANALYZE
    SELECT
        message_key,
        observation_index,
        message_sequence_index,
        sensor_index
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
      AND dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT :producer_batch_size
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
        "producer_batch_size": PRODUCER_BATCH_SIZE,
    },
)

display(producer_pickup_explain_dataframe)

,QUERY PLAN
0,Limit (cost=0.56..4416.20 rows=26000 width=76...
1,-> Index Scan using idx_synthetic_sensor_me...
2,Filter: ((producer_sent_at IS NULL) AN...
3,Planning Time: 0.334 ms
4,Execution Time: 25.063 ms


----

In [11]:
queue_status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

display(queue_status_dataframe)

,queue_status,row_count
0,pending,11700000


---

In [14]:
stage_7_readiness_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS total_queue_rows,
        COUNT(*) FILTER (WHERE queue_status = 'pending') AS pending_count,
        COUNT(*) FILTER (WHERE queue_status = 'claimed') AS claimed_count,
        COUNT(*) FILTER (WHERE queue_status = 'sent') AS sent_count,
        COUNT(*) FILTER (WHERE queue_status = 'failed') AS failed_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS populated_sent_at_count,
        COUNT(*) FILTER (WHERE producer_delivery_error IS NOT NULL) AS populated_error_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_7_readiness_dataframe)

row = stage_7_readiness_dataframe.iloc[0]

ready_for_stage_7 = (
    int(row["total_queue_rows"]) > 0
    and int(row["pending_count"]) == int(row["total_queue_rows"])
    and int(row["claimed_count"]) == 0
    and int(row["sent_count"]) == 0
    and int(row["failed_count"]) == 0
    and int(row["populated_claim_token_count"]) == 0
    and int(row["populated_sent_at_count"]) == 0
)

print("Ready for Stage 7:", ready_for_stage_7)

if not ready_for_stage_7:
    print("Queue is not clean. Use 06.5 repair/reset tools before running Stage 7.")

,total_queue_rows,pending_count,claimed_count,sent_count,failed_count,populated_claim_token_count,populated_sent_at_count,populated_error_count
0,11700000,11700000,0,0,0,0,0,0


Ready for Stage 7: True


----

In [15]:
# Dispose Engine
engine.dispose()